# Unidad 4 · Colab 3 de 3
## Aplicación práctica: microservicio de predicciones y métricas de negocio

**Objetivos de este notebook**

- Diseñar un microservicio FastAPI que expone predicciones y métricas para que las consuma una app web.
- Manejar errores HTTP apropiados (`404`, `422`, `500`) con `HTTPException`.
- Habilitar CORS para que un frontend en otro dominio pueda consumir la API.
- Conectar esto con lo visto en las Unidades 5 y 6: cómo se consume desde otra app y cómo se despliega.

> **Nivel:** intermedio. Este notebook integra todo lo visto en los Colab 1 y 2 de esta unidad.

---

## 1. Diseño del microservicio

Vamos a construir una API que un equipo de producto podría consumir desde una app web, con dos capacidades típicas de un microservicio de datos:

- `POST /predict` — recibe features de un cliente y devuelve una predicción (por ejemplo, probabilidad de abandono/churn).
- `GET /metrics` — devuelve métricas de negocio agregadas (por ejemplo, ventas por período), con filtros por query params.
- `GET /health` — endpoint de salud, típico para que la plataforma de despliegue (Unidad 6) sepa si el servicio está vivo.

## 2. Modelos de entrada y salida para la predicción

```python
from pydantic import BaseModel, Field
from typing import Literal

class ClienteFeatures(BaseModel):
    antiguedad_meses: int = Field(ge=0)
    gasto_mensual: float = Field(ge=0)
    reclamos_ultimo_trimestre: int = Field(ge=0)

class PrediccionOut(BaseModel):
    probabilidad_churn: float
    riesgo: Literal['bajo', 'medio', 'alto']
```

Separar `ClienteFeatures` (entrada) de `PrediccionOut` (salida) deja claro qué espera recibir el servicio y qué garantiza devolver — el contrato de la API.

In [ ]:
!pip install -q fastapi uvicorn

from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from typing import Literal

app = FastAPI(title='Microservicio de Predicciones', version='1.0.0')

class ClienteFeatures(BaseModel):
    antiguedad_meses: int = Field(ge=0)
    gasto_mensual: float = Field(ge=0)
    reclamos_ultimo_trimestre: int = Field(ge=0)

class PrediccionOut(BaseModel):
    probabilidad_churn: float
    riesgo: Literal['bajo', 'medio', 'alto']

def predecir_churn(features: ClienteFeatures) -> PrediccionOut:
    # Heuristica simple en lugar de un modelo entrenado, solo para el ejemplo
    score = 0.1 * features.reclamos_ultimo_trimestre - 0.01 * features.antiguedad_meses
    probabilidad = max(0.0, min(1.0, 0.3 + score))
    riesgo = 'alto' if probabilidad > 0.6 else 'medio' if probabilidad > 0.3 else 'bajo'
    return PrediccionOut(probabilidad_churn=round(probabilidad, 2), riesgo=riesgo)

@app.post('/predict', response_model=PrediccionOut)
def predict(features: ClienteFeatures):
    return predecir_churn(features)

cliente = TestClient(app)
resp = cliente.post('/predict', json={'antiguedad_meses': 3, 'gasto_mensual': 50, 'reclamos_ultimo_trimestre': 4})
print(resp.status_code, resp.json())

### Ejercicio 1 — Endpoint `/health`

Agregá un endpoint `GET /health` que devuelva `{'status': 'ok'}` con código `200`. Es el endpoint que las plataformas de la Unidad 6 (Render, Railway) usan para chequear que tu servicio sigue respondiendo.

<details>
<summary>💡 Ver solución</summary>

```python
@app.get('/health')
def health():
    return {'status': 'ok'}

resp = cliente.get('/health')
print(resp.status_code, resp.json())
```

</details>

## 3. Manejar errores explícitamente con `HTTPException`

Pydantic ya devuelve `422` automáticamente ante datos inválidos. Para otros errores de negocio (recurso no encontrado, reglas propias), usás `HTTPException`:

```python
from fastapi import HTTPException

METRICAS_DISPONIBLES = {'ventas', 'usuarios_activos', 'churn_promedio'}

@app.get('/metrics')
def metrics(nombre: str):
    if nombre not in METRICAS_DISPONIBLES:
        raise HTTPException(status_code=404, detail=f'Metrica {nombre} no encontrada')
    return {'nombre': nombre, 'valor': 12345}
```

### Ejercicio 2 — `GET /metrics` con filtros

Extendé el endpoint `/metrics` para que además reciba un query param opcional `periodo: str = 'mensual'`, valide que sea uno de `{'diario', 'semanal', 'mensual'}` (si no, `422` con `HTTPException`), y lo incluya en la respuesta.

<details>
<summary>💡 Ver solución</summary>

```python
PERIODOS_VALIDOS = {'diario', 'semanal', 'mensual'}

@app.get('/metrics')
def metrics(nombre: str, periodo: str = 'mensual'):
    if nombre not in METRICAS_DISPONIBLES:
        raise HTTPException(status_code=404, detail=f'Metrica {nombre} no encontrada')
    if periodo not in PERIODOS_VALIDOS:
        raise HTTPException(status_code=422, detail=f'Periodo {periodo} no valido')
    return {'nombre': nombre, 'periodo': periodo, 'valor': 12345}

resp = cliente.get('/metrics', params={'nombre': 'ventas', 'periodo': 'semanal'})
print(resp.status_code, resp.json())
```

</details>

## 4. CORS: permitir que una app web consuma tu API

Si tu API va a ser consumida desde un frontend en otro dominio (por ejemplo, una app en React en un dominio propio llamando a tu API en Render), el navegador bloquea la respuesta por CORS salvo que el servidor lo autorice explícitamente.

```python
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=['https://miapp.com'],
    allow_methods=['GET', 'POST'],
    allow_headers=['*'],
)
```

Documentación oficial: [CORS en FastAPI](https://fastapi.tiangolo.com/tutorial/cors/)

### Ejercicio 3 — Configurar CORS

¿Qué `allow_origins` configurarías si tu app web todavía no tiene dominio propio y estás probando localmente en `http://localhost:3000`? ¿Y si además querés que funcione desde cualquier origen mientras es un proyecto de curso?

<details>
<summary>💡 Ver solución</summary>

Para desarrollo local: `allow_origins=['http://localhost:3000']`. Para un proyecto de curso donde no importa restringir el origen: `allow_origins=['*']` — pero esto no se recomienda en producción real, porque cualquier sitio podría consumir tu API desde el navegador de un usuario.

</details>

## 5. De vuelta al resto del curso

- **Unidad 5:** el mismo patrón de `requests` que usaste para scrapear (Colab 1 de esa unidad) es el que usaría cualquier cliente — incluida tu propia app web — para consumir este microservicio.
- **Unidad 6:** este microservicio se dockeriza (mismo patrón `Dockerfile` con `uvicorn`) y se despliega en Render/Railway/HF Spaces igual que cualquier otra API FastAPI — el endpoint `/health` que armaste en el Ejercicio 1 es justamente lo que esas plataformas usan para confirmar que el deploy está sano.

## Mini-proyecto final: microservicio completo

Armá la versión final del microservicio con:

1. `POST /predict` — con el modelo Pydantic de entrada/salida y al menos una validación de negocio propia (por ejemplo, rechazar `antiguedad_meses` mayor a 600).
2. `GET /metrics` — con al menos dos filtros por query params.
3. `GET /health`.
4. CORS configurado.
5. `title`, `version` y `description` en la `app`, y `tags` en cada endpoint.
6. Al menos 4 pruebas con `TestClient` (casos exitosos y de error).

**Entregable:** el código completo de la API + las pruebas con `TestClient`. Como extensión opcional: dockerizala (Unidad 6) y desplegala en Render/Railway/HF Spaces.

## Autoevaluación

- [ ] Mi API valida los datos de entrada con Pydantic
- [ ] Uso `response_model` para controlar qué se expone en las respuestas
- [ ] Manejo errores de negocio con `HTTPException` y códigos apropiados
- [ ] Tengo un endpoint `/health`
- [ ] Configuré CORS
- [ ] La documentación en `/docs` describe claramente cada endpoint

---

**Fin de la Unidad 4.** Con estos tres notebooks recorriste el ciclo completo: consumir APIs externas, construir las tuyas con FastAPI y Pydantic, y aplicarlo a un microservicio real de predicciones y métricas — listo para las Unidades 5 y 6.